# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedshereef1/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

Turns validated model output into a human-reviewed content action playbook.
Sections run in order. Every claim is scoped to what the evidence supports.

> Skills loaded: `writing-honest-claims` + `flyrank/flyrank-data`

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### How the queue is built

The Random Forest model trained in w05 produces a probability score (`rf_proba`) for each content item — the estimated likelihood that the item carries the observed decline label (`is_declining_label = 1`). Items are ranked top-to-bottom by that score. The result is a priority queue: items at the top carry the most model-measured signals associated with decline in the training data.

**Archetype → Action mapping**

Four archetypes cover the decision space:

| Reason code | Signals present | Recommended action |
|---|---|---|
| `MODEL_HIGH_DECLINE` | rf_proba ≥ 0.70 | **Refresh content** — highest-confidence signal of decline; prioritise for editorial review and update |
| `MODEL_MOD_STALE` | rf_proba 0.50–0.70 AND days_since_last_update ≥ 104 | **Review freshness** — model is directionally cautious; staleness corroborates but does not confirm decline |
| `MODEL_MOD_VOLUME` | rf_proba 0.50–0.70 AND search_volume ≥ 20 | **Review opportunity** — decline signal is moderate; high search volume suggests the item is worth attention regardless |
| `MONITOR` | rf_proba < 0.50 | **Monitor** — model does not flag decline; re-check at next refresh cycle |

**Decay / refresh insight (observed, not causal)**

In this dataset, `days_with_impressions` and `log_impressions_90d` are the two strongest model features (permutation importance 0.115 and 0.034 respectively on the held-out test clients). Items with consistently low impression days over the 90-day window are more often associated with the decline label. This is observational: the pattern does not prove that refreshing content will reverse a decline, because the measurement is a static snapshot and we ran no intervention study. The honest framing is: *content with fewer impression-active days over 90 days was observed to co-occur with the decline label more often — these items appear worth reviewing first.*

CTR (0.017 permutation importance) is the third-ranked feature. Items with very low CTR despite having impression history show a similar co-occurrence pattern with decline.

In [ ]:
# ============================================================
# 1. SETUP, DATA LOAD, AND MODEL REPLICATION
# Mirrors w05_model.ipynb exactly — same seed, same split,
# same features — so rf_proba values are reproducible.
# ============================================================

import pandas as pd
import numpy as np
import json
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score

matplotlib.rcParams["figure.dpi"] = 120

RANDOM_SEED = 42

DATA_PATH = Path("../../data/raw/content_refresh_anonymized.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")

# Feature engineering (mirrors w05)
for col in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    df[f"log_{col}"] = np.log1p(df[col].fillna(0))

df["has_clicks"]      = (df["clicks_90d"] > 0).astype(int)
df["has_ai_sessions"] = (df["ai_sessions_90d"] > 0).astype(int)
df["measurable_opportunity"] = (
    (df["impressions_90d"] >= 100) & (df["sessions_90d"] > 0)
).astype(int)
df["avg_position"] = df["avg_position"].replace(0, np.nan).fillna(df["avg_position"].median())
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc",
    "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d",
    "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
    "has_clicks", "has_ai_sessions", "measurable_opportunity",
]
CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent",
    "age_tier", "freshness_tier", "word_count_tier",
    "impression_tier", "position_tier",
]

for col in NUMERIC_FEATURES:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
for col in CATEGORICAL_FEATURES:
    df[col] = df[col].fillna("unknown").astype(str)
    le = LabelEncoder()
    df[col + "_enc"] = le.fit_transform(df[col])

ENC_FEATURES = [c + "_enc" for c in CATEGORICAL_FEATURES]
ALL_FEATURES = NUMERIC_FEATURES + ENC_FEATURES

# Client-grouped split (same as w05)
rng = np.random.default_rng(RANDOM_SEED)
clients = df["client_id"].unique().to_numpy()
rng.shuffle(clients)
n_test_clients = max(1, int(len(clients) * 0.2))
test_clients  = set(clients[:n_test_clients])
train_clients = set(clients[n_test_clients:])

train_df = df[df["client_id"].isin(train_clients)].copy()
test_df  = df[df["client_id"].isin(test_clients)].copy()

X_train = train_df[ALL_FEATURES].values
y_train = train_df["is_declining_label"].values
X_test  = test_df[ALL_FEATURES].values
y_test  = test_df["is_declining_label"].values

print(f"Train: {len(train_df):,} rows | {len(train_clients)} clients")
print(f"Test:  {len(test_df):,} rows  | {len(test_clients)} clients")

# Train model
rf = RandomForestClassifier(
    n_estimators=200, max_depth=12, min_samples_leaf=10,
    random_state=RANDOM_SEED, n_jobs=-1,
)
rf.fit(X_train, y_train)
rf_proba_test = rf.predict_proba(X_test)[:, 1]

# Score on ALL rows (for the full playbook queue)
X_all = df[ALL_FEATURES].values
df["rf_proba"] = rf.predict_proba(X_all)[:, 1]

def precision_at_k(y_true, scores, k):
    idx = np.argsort(scores)[::-1][:k]
    return float(np.array(y_true)[idx].mean())

p50 = precision_at_k(y_test, rf_proba_test, 50)
p20 = precision_at_k(y_test, rf_proba_test, 20)
auc = roc_auc_score(y_test, rf_proba_test)
base_rate = y_test.mean()

print(f"\nModel metrics on held-out test clients:")
print(f"  Base rate:     {base_rate:.1%}")
print(f"  Precision@20:  {p20:.1%}")
print(f"  Precision@50:  {p50:.1%}")
print(f"  ROC-AUC:       {auc:.3f}")

In [ ]:
# ============================================================
# REASON CODES + RANKED QUEUE
# ============================================================

STALE_THRESH  = df["days_since_last_update"].quantile(0.75)  # 104 days
VOLUME_THRESH = df["search_volume"].quantile(0.75)           # 20

def assign_reason_code(row):
    p = row["rf_proba"]
    stale  = row["days_since_last_update"] >= STALE_THRESH
    volume = row["search_volume"] >= VOLUME_THRESH
    if p >= 0.70:
        return "MODEL_HIGH_DECLINE"
    elif p >= 0.50 and stale:
        return "MODEL_MOD_STALE"
    elif p >= 0.50 and volume:
        return "MODEL_MOD_VOLUME"
    else:
        return "MONITOR"

ACTION_MAP = {
    "MODEL_HIGH_DECLINE": "Refresh content",
    "MODEL_MOD_STALE":    "Review freshness",
    "MODEL_MOD_VOLUME":   "Review opportunity",
    "MONITOR":            "Monitor",
}

df["reason_code"]  = df.apply(assign_reason_code, axis=1)
df["action_label"] = df["reason_code"].map(ACTION_MAP)

ranked = (
    df.sort_values("rf_proba", ascending=False)
      .reset_index(drop=True)
)
ranked["priority_rank"] = ranked.index + 1

print("=" * 55)
print("ACTION QUEUE — reason code distribution")
print("=" * 55)
dist = ranked["reason_code"].value_counts()
for code, n in dist.items():
    print(f"  {code:<22} {n:>6,}  ({n/len(ranked)*100:.1f}%)")

print("\nTop 20 items in the ranked queue:")
display(
    ranked[[
        "priority_rank", "content_id", "content_type",
        "rf_proba", "days_since_last_update",
        "search_volume", "reason_code", "action_label"
    ]].head(20)
)

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use

**Who:** Content strategists and editorial teams at FlyRank client accounts who schedule periodic content reviews.

**What for:** The ranked queue is a decision-support tool — a starting point for human review, not a final verdict. It answers the question: "Given our portfolio of content, which items show the most signals associated with observed decline and therefore deserve attention first?"

**What it is not:** It is not an automated content management system. It does not decide what changes to make to specific pages. It does not replace understanding of the client's editorial context, brand voice, or competitive landscape.

---

### Validity limits — where the output stops being trustworthy

| Limit | Reason |
|---|---|
| **Single 90-day snapshot** | The dataset is one cross-sectional observation window per content item. The model captures co-occurrence patterns in this period only. Trend dynamics outside this window are not represented. |
| **32 pseudonymized clients** | The model was trained and tested on client patterns from this specific portfolio. Clients with substantially different content mixes, industries, or audience behaviours may not be well-represented. |
| **Client-grouped test: 6 held-out clients** | Precision@50 = 84% and Precision@20 = 90% were measured on 6 held-out clients from 32 total. The estimate carries variance from that small sample. These numbers describe performance on *this draw of clients*, not a proven floor for all future clients. |
| **Label is observational** | `is_declining_label` is derived from impression trend within the same 90-day snapshot. The model flags patterns observed alongside decline — it does not identify the cause of decline or guarantee that refreshing a flagged page will reverse it. |
| **feedly articles have higher error rate** | Feedly articles lack keyword data (search_volume, competition, cpc are zero-filled), giving the model fewer discriminating signals for that content type. Error rate for feedly articles was 26.9% vs 41.6% for keyword articles on the test set. Both are higher than the 39.1% base rate for keyword articles and require a calibrated human check. |
| **No time-series component** | The model does not model trajectory — a page that was declining but recovered within the 90 days looks the same as one still declining. |
| **Boundary zone is unreliable** | Items with predicted probability 0.40–0.60 (observed: 636 rows in the test set) are genuinely ambiguous — the model's signals do not resolve cleanly there. Human review is especially important in this range. |

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review checklist

Before acting on any flagged item, a reviewer should confirm:

1. **Is the page intentionally unchanged?** Some pages are evergreen or frozen by editorial policy. Staleness alone is not a problem if the page answers its intent correctly.
2. **Does the decline pattern hold in the live search console?** The model uses a 90-day trailing snapshot. Check whether the trend is continuing or has already reversed in the most recent 30 days.
3. **Is the page's content type one where the model underperforms?** Feedly articles (all-zero keyword features) have a measured 26.9% error rate on the test set — reviewer confidence should be lower for these.
4. **Is the probability score in the boundary zone (0.40–0.60)?** If yes, the recommendation is directional only. Do not action without additional evidence.
5. **Does the reason code match editorial intuition?** `MODEL_HIGH_DECLINE` on a recently-updated, high-traffic page should raise a flag — it may be a model error or a page with unusual signal patterns.
6. **Is refreshing feasible right now?** Capacity, dependencies, scheduled redirection work, or pending redesigns may make a refresh inadvisable even for genuinely declining pages.

---

### No-go list — what must NOT be automated

| Do not automate | Why |
|---|---|
| **Content deletion** | The model flags decline, not irrelevance. A declining page may still serve users or have backlink equity. Automated deletion from a model score is out of scope. |
| **URL redirects or slug changes** | These have cascading SEO and analytics consequences that the model has no awareness of. |
| **Triggering content rewrite workflows without review** | The model produces a ranked list; the decision to allocate editorial time requires human judgement about priority, cost, and strategy. |
| **Client communications about content performance** | The output is an internal decision-support tool. Sharing raw model scores with clients without editorial interpretation risks misrepresentation. |
| **Applying the same action to all items at one threshold** | A batch action on "everything above 0.70" would include ~16% false positives at Precision@50. Individual item review is required. |
| **Using boundary-zone items as confident signals** | Items with rf_proba 0.40–0.60 are genuinely unresolved. Treating them as high-confidence flags misstates the evidence. |

In [ ]:
# ============================================================
# 3. BOUNDARY ZONE + FALSE POSITIVE ILLUSTRATION
# Shows the reviewer how many items fall in the uncertain range
# and what real false positives look like.
# ============================================================

test_df = test_df.copy()
test_df["rf_proba"] = rf_proba_test
test_df["rf_pred"]  = (rf_proba_test >= 0.5).astype(int)
test_df["correct"]  = (test_df["rf_pred"] == test_df["is_declining_label"]).astype(int)

boundary = test_df[test_df["rf_proba"].between(0.40, 0.60)]
print(f"Boundary-zone items (prob 0.40-0.60) in test set: {len(boundary):,}")
print(f"  ({len(boundary)/len(test_df)*100:.1f}% of test rows — human review essential here)")

# False positives in the top-50 flagged
top50_idx = np.argsort(rf_proba_test)[::-1][:50]
top50 = test_df.iloc[top50_idx].copy()
fp = top50[top50["is_declining_label"] == 0]

print(f"\nFalse positives in top-50 (items flagged 'Refresh' that are NOT declining): {len(fp)}")
print(f"  This is the {100-p50*100:.0f}% the Precision@50 = {p50:.0%} already accounts for.")
display(
    fp[[
        "content_id", "content_type",
        "days_since_last_update", "ctr", "rf_proba", "is_declining_label"
    ]].head(5).reset_index(drop=True)
)
print("Pattern: model assigned high proba but label = 0 (not declining).")
print("These require the reviewer to override the model recommendation.")

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring plan

The model was trained on one 90-day snapshot of 30,000 content items from 32 clients. It has no live data connection. The following signals indicate the model or queue needs review:

**Retrain triggers:**

| Signal | Threshold | Why it matters |
|---|---|---|
| Precision@50 on a fresh held-out sample drops below 65% | Below the base rate + meaningful margin | Model no longer ranks declining items meaningfully above chance |
| Label rate in new data shifts by more than ±10 percentage points | e.g. base rate moves from 54% to <44% or >64% | Distribution shift — model learned patterns from a different regime |
| New major content type introduced (not in training data) | Any new category with >500 items | Model has no learned signal for that type; scores are extrapolations |
| Top feature distributions shift significantly | days_with_impressions or log_impressions_90d median shifts >20% | The key features the model relies on are no longer measured the same way |
| New data vintage available (next 90-day snapshot) | Every ~90 days by nature | Refreshing the snapshot keeps the staleness/trend signals accurate |

**Monitoring cadence:**

- **Immediate:** After any editorial intervention on a top-50 flagged item, record whether the flag matched the editorial team's assessment (precision feedback loop).
- **Quarterly:** Re-compute precision on the newest snapshot against the current model. If it drops below 65%, schedule retraining.
- **On new client onboarding:** Run a descriptive check that the new client's content type and volume distribution falls within the training distribution. Flag if it does not.

**What healthy looks like:**
Editorial teams report that top-20 flagged items match their review priority at least 70% of the time (editorial agreement rate ≥ 70%). The model is a starting point — human override should be expected and is not a failure mode; it is evidence the human review step is working.

In [ ]:
# ============================================================
# 4. MONITORING REFERENCE NUMBERS
# Save the key metrics to a JSON for the paper and retrain checks.
# ============================================================

metrics = {
    "model": "RandomForest (n_estimators=200, max_depth=12, min_samples_leaf=10)",
    "split": "client-grouped 80/20",
    "train_clients": len(train_clients),
    "test_clients": len(test_clients),
    "train_rows": int(len(train_df)),
    "test_rows": int(len(test_df)),
    "base_rate_test": round(float(base_rate), 4),
    "precision_at_20": round(float(p20), 4),
    "precision_at_50": round(float(p50), 4),
    "roc_auc": round(float(auc), 4),
    "retrain_trigger_precision_threshold": 0.65,
    "top_features": [
        {"feature": "days_with_impressions",  "importance_mean": 0.1147},
        {"feature": "log_impressions_90d",    "importance_mean": 0.0345},
        {"feature": "ctr",                    "importance_mean": 0.0167},
    ],
    "label_source": "is_declining_label derived from trend_direction (= trend_pct < 0)",
    "data_vintage": "90-day snapshot, single cross-section",
    "snapshot_date_note": "data/raw/content_refresh_anonymized.csv — trailing 90 days",
}

METRICS_PATH = Path("work/outputs/playbook_metrics.json")
METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(METRICS_PATH, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Metrics saved to {METRICS_PATH}")
print(json.dumps(metrics, indent=2))

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
# ============================================================
# 5A. EXPORT THE RANKED QUEUE CSV
# content_id is a pseudonym — safe to include.
# No raw client names, URLs, or private queries.
# ============================================================

QUEUE_COLS = [
    "priority_rank", "content_id", "content_type",
    "rf_proba", "days_since_last_update", "search_volume",
    "ctr", "days_with_impressions", "log_impressions_90d",
    "reason_code", "action_label",
]

queue_df = ranked[QUEUE_COLS].copy()
queue_df["rf_proba"] = queue_df["rf_proba"].round(4)

QUEUE_PATH = Path("work/outputs/action_playbook_queue.csv")
QUEUE_PATH.parent.mkdir(parents=True, exist_ok=True)
queue_df.to_csv(QUEUE_PATH, index=False)
print(f"Queue saved: {QUEUE_PATH}  ({len(queue_df):,} rows)")
print(f"  Columns: {list(queue_df.columns)}")
print("\nTop-5 preview:")
display(queue_df.head(5))

In [ ]:
# ============================================================
# 5B. FIGURE 1 — Model vs baseline comparison bar chart
# Saved to work/figures/ for paper reuse.
# ============================================================

FIGURES_PATH = Path("work/figures")
FIGURES_PATH.mkdir(parents=True, exist_ok=True)

# Validated numbers from w05 and w06
models = ["Base rate", "Rule baseline", "Logistic Regression", "Random Forest"]
p20_vals = [0.391, 0.300, 0.250, 0.900]
p50_vals = [0.391, 0.280, 0.360, 0.840]

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 4))
bars1 = ax.bar(x - width/2, p20_vals, width, label="Precision@20", color="#4C72B0", alpha=0.85)
bars2 = ax.bar(x + width/2, p50_vals, width, label="Precision@50", color="#DD8452", alpha=0.85)

ax.axhline(0.391, color="gray", linestyle="--", linewidth=0.9, label="Base rate (39.1%)")

ax.set_ylabel("Precision")
ax.set_title("Model vs Baseline — Precision on Held-Out Test Clients\n(6 clients, client-grouped split)")
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=12, ha="right")
ax.set_ylim(0, 1.05)
ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(xmax=1, decimals=0))
ax.legend(loc="upper left")

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.015,
            f"{bar.get_height():.0%}", ha="center", va="bottom", fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.015,
            f"{bar.get_height():.0%}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
fig.savefig(FIGURES_PATH / "fig1_model_vs_baseline.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Figure saved: {FIGURES_PATH / 'fig1_model_vs_baseline.png'}")

In [ ]:
# ============================================================
# 5C. FIGURE 2 — Reason code distribution by content type
# Shows how the queue breaks down across archetypes.
# ============================================================

# Reason code breakdown by content type
ct_breakdown = (
    ranked.groupby(["content_type", "reason_code"], observed=True)
          .size()
          .unstack(fill_value=0)
)

order = ["MODEL_HIGH_DECLINE", "MODEL_MOD_STALE", "MODEL_MOD_VOLUME", "MONITOR"]
order = [c for c in order if c in ct_breakdown.columns]
ct_breakdown = ct_breakdown[order]

colors = ["#c0392b", "#e67e22", "#2980b9", "#95a5a6"]

fig2, ax2 = plt.subplots(figsize=(7, 4))
ct_breakdown.plot(kind="bar", ax=ax2, color=colors[:len(order)], alpha=0.85, edgecolor="white")
ax2.set_title("Action Queue — Reason Code Distribution by Content Type")
ax2.set_ylabel("Number of content items")
ax2.set_xlabel("")
ax2.tick_params(axis="x", rotation=15)
ax2.legend(title="Reason code", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
plt.tight_layout()
fig2.savefig(FIGURES_PATH / "fig2_reason_code_by_content_type.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Figure saved: {FIGURES_PATH / 'fig2_reason_code_by_content_type.png'}")

In [ ]:
# ============================================================
# 5D. FIGURE 3 — Score distribution (rf_proba histogram)
# Shows the spread of model confidence across all content items.
# ============================================================

fig3, ax3 = plt.subplots(figsize=(7, 3.5))
ax3.hist(ranked["rf_proba"], bins=40, color="#4C72B0", alpha=0.8, edgecolor="white")
ax3.axvline(0.70, color="#c0392b", linestyle="--", linewidth=1.2, label="High-decline threshold (0.70)")
ax3.axvline(0.50, color="#e67e22", linestyle="--", linewidth=1.2, label="Action threshold (0.50)")
ax3.set_xlabel("Model-estimated decline probability (rf_proba)")
ax3.set_ylabel("Number of content items")
ax3.set_title("Distribution of Decline Probability Scores — All 30,000 Items")
ax3.legend(fontsize=8)
plt.tight_layout()
fig3.savefig(FIGURES_PATH / "fig3_score_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Figure saved: {FIGURES_PATH / 'fig3_score_distribution.png'}")

In [ ]:
# ============================================================
# 5E. COST / VALUE FRAMING
# ============================================================

high_decline_n = int((ranked["reason_code"] == "MODEL_HIGH_DECLINE").sum())
mod_stale_n    = int((ranked["reason_code"] == "MODEL_MOD_STALE").sum())
mod_volume_n   = int((ranked["reason_code"] == "MODEL_MOD_VOLUME").sum())
monitor_n      = int((ranked["reason_code"] == "MONITOR").sum())

print("=" * 60)
print("COST / VALUE FRAMING")
print("=" * 60)
print(f"""
Across all {len(ranked):,} content items:

  MODEL_HIGH_DECLINE  {high_decline_n:>6,}  items  — Refresh content (highest priority)
  MODEL_MOD_STALE     {mod_stale_n:>6,}  items  — Review freshness
  MODEL_MOD_VOLUME    {mod_volume_n:>6,}  items  — Review opportunity
  MONITOR             {monitor_n:>6,}  items  — No immediate action needed

At Precision@50 = {p50:.0%} (held-out test clients), roughly
{(1-p50)*50:.0f} of every 50 'Refresh content' flags are expected
to be false positives — items the model scores high that are not
actually declining. Human review at the item level catches these.

Value proposition (honest framing):
  Without the model, a reviewer scanning {len(ranked):,} items randomly
  would find a declining item {base_rate:.0%} of the time (base rate).
  With the model's top-50 queue, the observed rate on held-out clients
  was {p50:.0%} — a {p50/base_rate:.1f}x improvement over random selection
  on those six clients. This is measured, not guaranteed.
""")

In [ ]:
# ============================================================
# 5F. VERIFY ALL EXPORTS EXIST
# ============================================================

exports = [
    QUEUE_PATH,
    METRICS_PATH,
    FIGURES_PATH / "fig1_model_vs_baseline.png",
    FIGURES_PATH / "fig2_reason_code_by_content_type.png",
    FIGURES_PATH / "fig3_score_distribution.png",
]

print("Export verification:")
all_ok = True
for p in exports:
    exists = p.exists()
    status = "OK " if exists else "MISSING"
    print(f"  [{status}] {p}")
    if not exists:
        all_ok = False

if all_ok:
    print("\n✓ All exports present.")
else:
    print("\n✗ Some exports are missing — re-run from the top.")

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.